In [ ]:
##INITIAL CURRICULUM TRAINING

import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

DATA_DIR = "/mnt/beegfs/scoulombe"
SNAPSHOT_TAG = "binned_finegrained_v2_bigmodel_n64_v1"   # <-- only required change from the bigmodel notebook
BIN_NAMES = ["0", "1", "2"]
N_CURRICULUM_STAGES = 26

TRAIN_HISTORY_FILE = os.path.join(DATA_DIR, f"train_history_{SNAPSHOT_TAG}.npz")
RESUME_STATE_FILE   = os.path.join(DATA_DIR, f"resume_state_{SNAPSHOT_TAG}.npz")

SKIP_FIRST_N = 1  # number of leading epochs to omit from the plots

if not os.path.exists(TRAIN_HISTORY_FILE):
    print(f"[missing] {TRAIN_HISTORY_FILE} -- no training history to plot yet.")
else:
    hist = np.load(TRAIN_HISTORY_FILE, allow_pickle=True)
    train_losses = hist["train_losses"]
    val_losses   = hist["val_losses"]
    best_val_loss = float(hist["best_val_loss"])
    best_epoch    = hist["best_epoch"]
    best_epoch    = None if best_epoch is None or (np.ndim(best_epoch) == 0 and best_epoch.item() is None) else int(best_epoch)

    stage_log = []
    if os.path.exists(RESUME_STATE_FILE):
        rs = np.load(RESUME_STATE_FILE, allow_pickle=True)
        stage_log = list(rs["stage_log"])
    else:
        print(f"[note] {RESUME_STATE_FILE} not found -- plotting raw loss curves only "
              f"(expected to exist for this script, since it does call save_resume_state; "
              f"if missing, training may not have progressed past stage 1 yet).")

    print(f"{len(train_losses)} epochs recorded so far.")
    print(f"Best combined val_loss = {best_val_loss:.5f} at epoch {best_epoch}")
    if stage_log:
        latest_stage_idx = int(stage_log[-1][1])
        print(f"Currently in (or last recorded) stage {latest_stage_idx + 1}/{N_CURRICULUM_STAGES}")

    # --- combined (weighted) loss ---
    fig, ax = plt.subplots(figsize=(14, 6))
    epochs = np.arange(1, len(train_losses) + 1)
    plot_mask = epochs > SKIP_FIRST_N

    ax.plot(epochs[plot_mask], train_losses[plot_mask], label="Train", linewidth=1.5)
    ax.plot(epochs[plot_mask], val_losses[plot_mask], label="Test", linewidth=1.5)

    first_boundary_label_done = False
    for entry in stage_log[1:]:
        if entry[0] > SKIP_FIRST_N:
            ax.axvline(entry[0], color='gray', linestyle='--', linewidth=1, alpha=0.6,
                       label="Stage Transition" if not first_boundary_label_done else None)
            first_boundary_label_done = True

    # --- per-stage best-epoch markers + discarded (rolled-back) regions ---
    def compute_stage_segments(stage_log, n_epochs):
        segments = []
        for i, entry in enumerate(stage_log):
            start = int(entry[0])
            stage_idx = int(entry[1])
            end = int(stage_log[i+1][0]) if i+1 < len(stage_log) else n_epochs
            segments.append({"stage_idx": stage_idx, "start": start, "end": end,
                              "is_last_segment": (i+1 >= len(stage_log))})
        return segments

    if stage_log:
        segments = compute_stage_segments(stage_log, len(val_losses))
        first_stage_label_done = False
        first_discard_label_done = False
        for seg in segments:
            s, e = seg["start"], seg["end"]
            if e <= s:
                continue  # degenerate/zero-length segment, nothing to show
            seg_vals = val_losses[s:e]
            best_local = int(np.argmin(seg_vals))
            stage_best_epoch = s + best_local + 1
            discarded_start = stage_best_epoch + 1
            discarded_end = e

            # marker at this stage's true best epoch (what STAGE_BEST_CHECKPOINT_FILE pointed to)
            if stage_best_epoch > SKIP_FIRST_N:
                ax.plot(stage_best_epoch, val_losses[stage_best_epoch - 1], marker='*', markersize=14,
                         color='darkorange', zorder=5,
                         label="Stage Best" if not first_stage_label_done else None)
                first_stage_label_done = True

            # shade the discarded range, if any epochs actually got thrown away
            if discarded_end > discarded_start - 1:
                shade_start = max(discarded_start - 0.5, SKIP_FIRST_N + 0.5)
                if discarded_end + 0.5 > shade_start:
                    ax.axvspan(shade_start, discarded_end + 0.5, color='red', alpha=0.08,
                               label="Discarded" if not first_discard_label_done else None)
                    first_discard_label_done = True

    ax.set_xlabel("Epoch")
    ax.set_ylabel("Total Loss")
    ax.set_xlim(left=SKIP_FIRST_N + 0.5)
    ax.legend(loc='upper right')
    plt.tight_layout()

    combined_path = "/tmp/_combined_loss_preview.png"
    plt.savefig(combined_path, dpi=100)
    plt.close(fig)
    display(Image(combined_path))

    # --- per-component breakdown (pixel / patch / spectral), train vs val ---
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    components = [
        ("Pixel", hist["train_losses_pixel"], hist["val_losses_pixel"]),
        ("Patch", hist["train_losses_patch"], hist["val_losses_patch"]),
        ("Spectral", hist["train_losses_spectral"], hist["val_losses_spectral"]),
    ]
    for ax, (name, train_c, val_c) in zip(axes, components):
        ax.plot(epochs[plot_mask], train_c[plot_mask], label="Train", linewidth=1.2)
        ax.plot(epochs[plot_mask], val_c[plot_mask], label="Test", linewidth=1.2)
        first_boundary_label_done_sub = False
        for entry in stage_log[1:]:
            if entry[0] > SKIP_FIRST_N:
                ax.axvline(entry[0], color='gray', linestyle='--', linewidth=0.8, alpha=0.5,
                           label="Stage Transition" if not first_boundary_label_done_sub else None)
                first_boundary_label_done_sub = True
        ax.set_ylabel(f" {name} Loss")
        ax.set_xlabel("Epoch")
        ax.set_xlim(left=SKIP_FIRST_N + 0.5)
        ax.legend(loc='upper right')
    plt.tight_layout()

    components_path = "/tmp/_component_loss_preview.png"
    plt.savefig(components_path, dpi=100)
    plt.close(fig)
    display(Image(components_path))

##FINETUNING

DATA_DIR = "/mnt/beegfs/scoulombe"

# This must match SNAPSHOT_TAG from the training script for the run you're analyzing.
SNAPSHOT_TAG = os.environ.get("SNAPSHOT_TAG", "binned_finetune_flamingo_rot_v1")

TRAIN_HISTORY_FILE  = os.path.join(DATA_DIR, f"train_history_{SNAPSHOT_TAG}.npz")
RESUME_STATE_FILE   = os.path.join(DATA_DIR, f"resume_state_{SNAPSHOT_TAG}.npz")
CHECKPOINT_FILE      = os.path.join(DATA_DIR, f"model_weights_{SNAPSHOT_TAG}.pt")
BEST_CHECKPOINT_FILE = os.path.join(DATA_DIR, f"model_weights_{SNAPSHOT_TAG}_best.pt")

for label, path in [("train history", TRAIN_HISTORY_FILE),
                     ("resume state", RESUME_STATE_FILE),
                     ("checkpoint", CHECKPOINT_FILE),
                     ("best checkpoint", BEST_CHECKPOINT_FILE)]:
    print(f"{'FOUND   ' if os.path.exists(path) else 'MISSING '} {label}: {path}")

assert os.path.exists(TRAIN_HISTORY_FILE), f"Train history file not found: {TRAIN_HISTORY_FILE}"

hist = np.load(TRAIN_HISTORY_FILE)

train_losses = hist["train_losses"]
val_losses   = hist["val_losses"]

train_losses_pixel    = hist["train_losses_pixel"]
train_losses_patch    = hist["train_losses_patch"]
train_losses_spectral = hist["train_losses_spectral"]

val_losses_pixel    = hist["val_losses_pixel"]
val_losses_patch    = hist["val_losses_patch"]
val_losses_spectral = hist["val_losses_spectral"]

weight_pixel    = float(hist["weight_pixel"])
weight_patch    = float(hist["weight_patch"])
weight_spectral = float(hist["weight_spectral"])

best_val_loss = float(hist["best_val_loss"])
best_epoch    = int(hist["best_epoch"])

n_epochs = len(train_losses)
epochs = np.arange(1, n_epochs + 1)

print(f"Epochs logged: {n_epochs}")
print(f"Best combined val_loss: {best_val_loss:.6f} at epoch {best_epoch}")
print(f"Loss weights -> pixel={weight_pixel:.4e}  patch={weight_patch:.4e}  spectral={weight_spectral:.4e}")

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(epochs, train_losses, label="Train", color="tab:blue")
ax.plot(epochs, val_losses, label="Test", color="darkorange")
if best_epoch is not None and best_epoch > 0 and best_epoch < n_epochs:
    ax.axvspan(best_epoch + 0.5, n_epochs + 0.5, color='red', alpha=0.08,
               label="Discarded")
if best_epoch is not None and best_epoch > 0:
    ax.plot(best_epoch, val_losses[best_epoch - 1], marker='*', markersize=16,
             color="darkorange", zorder=5, label=f"Stage Best")
ax.set_xlabel("Epoch")
ax.set_ylabel("Total Loss")
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharex=True)
component_data = [
    ("Pixel", train_losses_pixel, val_losses_pixel),
    ("Patch", train_losses_patch, val_losses_patch),
    ("Spectral", train_losses_spectral, val_losses_spectral),
]
for ax, (name, tr, va) in zip(axes, component_data):
    ax.plot(epochs, tr, label="Train", color="tab:blue")
    ax.plot(epochs, va, label="Test", color="tab:orange")
    ax.set_ylabel(f"{name} Loss")
    ax.set_xlabel("Epoch")
    ax.legend(loc='upper right')
axes[0].set_ylabel("loss (log scale)")
plt.tight_layout()
plt.show()